In [10]:
import numpy as np
#system config
N=3
np.random.seed(14)
W=np.random.randn(N,N)*0.1
b=np.random.randn(N)

To a simple 3 neuron RNN!!

In [11]:
x=np.zeros(N)
def F(x):
  return -x + np.tanh(W @ x + b)

#free phase
def free_phase(x):
  steps=5000
  dt=0.05
  for i in range(steps):
    dx=F(x)
    x=x+(dx*dt)
    if np.linalg.norm(dx) < 1e-5:
        break
  return x #x{0}_bar

x_bar_0= free_phase(x)
print("Stationary state x_bar^0:\n", free_phase(x_bar_0))

Stationary state x_bar^0:
 [ 0.83704378 -0.16036596 -0.1079255 ]


In [12]:
#jacobian decomposition A= 0.5*(J-J^T)
J=np.zeros((N,N))
for i in range(N):
  z_i=np.dot(W[i],x_bar_0)+b[i]
  d_tanhx=1.0-np.tanh(z_i)**2
  for j in range (N):
    leak_term=-1.0 if (i==j) else 0
    interaction_term= np.dot(d_tanhx,W[i,j])
    J[i,j]=leak_term+interaction_term

A = 0.5 * (J - J.T)
print(A)
print()
print(J)

[[ 0.          0.00470902  0.07678803]
 [-0.00470902  0.         -0.00338465]
 [-0.07678803  0.00338465  0.        ]]

[[-0.953561    0.00237042  0.00520795]
 [-0.00704763 -1.19527832  0.01409571]
 [-0.14836811  0.02086502 -1.05517024]]


In [13]:
y_target = np.array([1.0, -1.0, 0.5]) #target desired
#cost func :
C=0.5*np.linalg.norm(y_target-free_phase(x_bar_0))**2
print(C)

0.5505567298177806


In [23]:
#defining cost gradient
def cost_gradient_func(x):
  return x-y_target
def evolve_nudged_phase(beta,steps=5000,dt=0.05,thresh=1e-5):
  x=x_bar_0.copy() #starting from the equilibrium position obtained from phase 1
  for _ in range(steps):
    dx=(F(x) - beta * cost_gradient_func(x) - 2.0 * (A @ (x - x_bar_0))) * dt
    x=x+dx
    if np.linalg.norm(dx) < thresh:
      break
  return x
beta=0.1
x_pos = evolve_nudged_phase(beta=+beta)  # x_bar_A^(+beta)
x_neg = evolve_nudged_phase(beta=-beta)  # x_bar_A^(-beta)

print("Nudged state (+beta):\n", np.round(x_pos, 4))
print("Nudged state (-beta):\n", np.round(x_neg, 4))

Nudged state (+beta):
 [ 0.8452 -0.2139 -0.0487]
Nudged state (-beta):
 [ 0.8278 -0.08   -0.1698]


In [17]:
#parameter update --> to understand : When the network was pulled toward the right answer,
#did this wire carry more current or less current than when it was pulled away?
eta=0.05 #learning rate
#for positive nudge
z_pos = W @ x_pos + b
d_sigma_pos = 1.0 - np.tanh(z_pos)**2
#for negative nudge
z_neg = W @ x_neg + b
d_sigma_neg = 1.0 - np.tanh(z_neg)**2
#outer pdt to compute pairewise synaptic activations
dW_pos = np.outer(d_sigma_pos, x_pos)
dW_neg = np.outer(d_sigma_neg, x_neg)
#contrasive finite difference updates
Delta_W = (eta / (2.0 * beta)) * (dW_pos - dW_neg)
Delta_b = (eta / (2.0 * beta)) * (d_sigma_pos - d_sigma_neg)
#applying updates to the parameters
W += Delta_W
b += Delta_b
print("Updated Weights:\n", np.round(W, 5))
print("Updated Biases:\n", np.round(b, 5))



Updated Weights:
 [[ 0.15608 -0.00244  0.026  ]
 [-0.001   -0.2347   0.04201]
 [-0.14628 -0.01321 -0.02753]]
Updated Biases:
 [ 1.08408 -0.18397  0.01403]


In [26]:
def run_free_phase(W_curr, b_curr, dt=0.05, steps=5000, tol=1e-5):
    x = np.zeros(N)
    for _ in range(steps):
        dx = (-x + np.tanh(W_curr @ x + b_curr)) * dt
        x += dx
        if np.linalg.norm(dx) < tol:
            break
    return x

In [35]:
# Check new free equilibrium
x_bar_0_new = run_free_phase(W, b)
new_cost = 0.5 * np.linalg.norm(y_target - x_bar_0_new)**2

# Re-calculate original cost to ensure numerical value
original_cost_value = 0.5 * np.linalg.norm(y_target - free_phase(x_bar_0))**2

print("Original cost:", np.round(original_cost_value, 5))
print("New cost:     ", np.round(new_cost, 5))
print("Improvement:  ", np.round(original_cost_value - new_cost, 5))

Original cost: 0.55453
New cost:      0.55449
Improvement:   5e-05


In [41]:
import numpy as np

np.random.seed(42)
N = 3

W = np.random.randn(N, N) * 0.3
b = np.random.randn(N) * 0.5
y_target = np.array([1.0, -1.0, 0.5])

def F(x, W_curr, b_curr):
    return -x + np.tanh(W_curr @ x + b_curr)

def cost(x):
    return 0.5 * np.sum((x - y_target)**2)

def grad_C(x):
    return x - y_target

def run_free_phase(W_curr, b_curr, dt=0.05, steps=5000, tol=1e-5):
    x = np.zeros(N)
    for _ in range(steps):
        dx = F(x, W_curr, b_curr) * dt
        x += dx
        if np.linalg.norm(dx) < tol:
            break
    return x

def run_nudged_phase(beta, x_eq, W_curr, b_curr, A_J_curr, dt=0.05, steps=5000, tol=1e-5):
    x = x_eq.copy()
    for _ in range(steps):
        dx = (F(x, W_curr, b_curr) - beta * grad_C(x) - 2.0 * (A_J_curr @ (x - x_eq))) * dt
        x += dx
        if np.linalg.norm(dx) < tol:
            break
    return x

eta = 0.1   # Learning rate
beta = 0.1

print(f"{'Step':>4} | {'Cost':>8} | {'Free State x':>28}")
print("-" * 48)

for step in range(50):
    # 1. Free Phase
    x_bar_0 = run_free_phase(W, b)
    current_loss = cost(x_bar_0)

    # 2. Jacobian & Antisymmetric part
    J_F = np.zeros((N, N))
    for i in range(N):
        z_i = np.dot(W[i], x_bar_0) + b[i]
        d_sigma = 1.0 - np.tanh(z_i)**2
        for j in range(N):
            leak = -1.0 if (i == j) else 0.0
            J_F[i, j] = leak + d_sigma * W[i, j]
    A_J = 0.5 * (J_F - J_F.T)

    # 3. Nudged Phases
    x_pos = run_nudged_phase(+beta, x_bar_0, W, b, A_J)
    x_neg = run_nudged_phase(-beta, x_bar_0, W, b, A_J)

    # 4. Parameter Update (NEGATIVE sign for true gradient descent)
    z_pos = W @ x_pos + b
    d_sigma_pos = 1.0 - np.tanh(z_pos)**2

    z_neg = W @ x_neg + b
    d_sigma_neg = 1.0 - np.tanh(z_neg)**2

    dW_pos = np.outer(d_sigma_pos, x_pos)
    dW_neg = np.outer(d_sigma_neg, x_neg)

    # ---> MINUS SIGN HERE: (dW_neg - dW_pos) pushes downhill <---
    Delta_W = (eta / (2.0 * beta)) * (dW_neg - dW_pos)
    Delta_b = (eta / (2.0 * beta)) * (d_sigma_neg - d_sigma_pos)

    W += Delta_W
    b += Delta_b

    if (step + 1) % 10 == 0 or step == 0:
        print(f"{step+1:4d} | {current_loss:8.5f} | {np.round(x_bar_0, 3)!s:>28}")

x_final = run_free_phase(W, b)
print("-" * 48)
print("Target:    ", y_target)
print("Final Free:", np.round(x_final, 4))
print("Final Cost:", round(cost(x_final), 5))

Step |     Cost |                 Free State x
------------------------------------------------
   1 |  0.85046 |       [ 0.29  -0.086 -0.101]
  10 |  0.93776 |       [ 0.129 -0.175 -0.16 ]
  20 |  1.06002 |       [ 0.03  -0.05  -0.027]
  30 |  1.11758 |          [0.007 0.002 0.004]
  40 |  1.12738 |       [-0.003 -0.003 -0.004]
  50 |  1.12844 |       [-0.006 -0.006 -0.006]
------------------------------------------------
Target:     [ 1.  -1.   0.5]
Final Free: [-0.0063 -0.0061 -0.0064]
Final Cost: 1.12841
